## Aeropulse — Orchestration: 01 Identify Next Batch

**Purpose:** Finds the oldest `PENDING` batch for a given source and hands it back to the pipeline. Called at the start of the batch loop, and again at the end of every iteration to check whether another batch is waiting.

**Parameters (pipeline-injected):** `source_name`

**Returns** (via `mssparkutils.notebook.exit`): `{"has_next_batch": bool, "batch_id": str, "batch_year": str}`


In [ ]:
# Parameters
# Injected by the pipeline's Notebook activity base parameters — the default below is only
# used when this notebook is run standalone for testing (this is what was missing before:
# running the cell directly threw NameError: name 'source_name' is not defined).
source_name = "flight"


In [1]:
# 01 - identify next batch
import json

# find the oldest still-pending batch for this source; ORDER BY batch_id relies on the
# 'YYYY_MM' naming sorting correctly as plain text, which it does here
row = spark.sql(f"""
    SELECT batch_id, batch_year FROM control.batch_control
    WHERE source_name = '{source_name}' AND status = 'PENDING'
    ORDER BY batch_id LIMIT 1
""").collect()

# build the result the pipeline branches on: has_next_batch drives the Until loop's exit condition
result = (
    {"has_next_batch": True, "batch_id": row[0]["batch_id"], "batch_year": row[0]["batch_year"]}
    if row
    else {"has_next_batch": False}
)

# hand the result back to the calling pipeline activity, readable as output.result.exitValue
mssparkutils.notebook.exit(json.dumps(result))   # mssparkutils is available by default, no import needed


StatementMeta(, 7c540ac7-5dc3-468a-9843-1e8b1cd92728, 3, Finished, Available, Finished, True)

NameError: name 'source_name' is not defined